### Build the African Language Confusion Prompt Set

This notebook builds the prompt dataset for the study

| Source | Task | Languages | Target count/language |
|---|---|---|---|
| Aya | monolingual | 10 | 100 (curated from a 200-candidate pool) |
| Dolly | monolingual | 8 | 150 (curated from a 250-candidate pool) |
| PolyWrite | monolingual | 18 | 100 |
| ShareGPT | crosslingual | 18 (union of all monolingual languages) | 200 (curated from a 400-candidate pool) |

**Monolingual** = a prompt written natively in the target language, expecting a reply in
that language. **Crosslingual** = an English instruction asking the model to reply in a
target African language (built by re-templating English base prompts, substituting the
target language name into instruction patterns like `"Reply in {language}."` /
`"... Write in {language}."`).

**AfriQA and Okapi are currently disabled** (commented out of `LOADERS` in `prompts.py`):
AfriQA pending a decision on how to handle it, and Okapi because we're no longer using
it as a crosslingual source. Their loader functions are still there if we want to
re-enable either later -- see the module notes in `prompts.py`.

**Aya, Dolly and ShareGPT use a two-step candidate/curation process.** These sources turned
out to be noisy enough that a plain word-count filter wasn't sufficient, so instead of
sampling straight to the final target count, we now: (1) sample a larger pool of
candidates automatically, (2) you manually review that pool and prune out the prompts
you don't want, saving the pruned result to a "curated" path, and (3) the actual build
step below reads only from that curated file. See the **Generate candidate pools**
section below for the commands, and `generate_aya_candidates` / `generate_dolly_candidates` /
`generate_sharegpt_candidates` in `prompts.py` for the filtering logic (Aya: English-only +
5-20 words; Dolly: no "Context:" block + >=5 words; ShareGPT: English-only + 5-20 words).

**Note on Aya coverage:** Fon, Twi and Kinyarwanda are under Aya, but the
`CohereForAI/aya_dataset` split used here does not
contain any rows for those three languages -- confirmed by inspecting the full
`language` value_counts (73 languages, none of them Fon/Twi/Kinyarwanda). They exist only
in the larger, machine-templated `aya_collection_language_split`, which is a different
kind of data (templated NLP tasks vs. organic human prompts) and was deliberately not
mixed in. Fon, Twi and Kinyarwanda are still covered by PolyWrite and by the crosslingual
set -- see `languages.py` for the full registry and rationale.

Output: one CSV per (task, source, language) under `prompts/`, plus a single consolidated `prompts/all_prompts.csv`
with columns `id, prompt, source, task, language`. The `Open_Source_Models.ipynb` and
`Closed_Source_Models.ipynb` notebooks read `all_prompts.csv` to run the model sweeps.

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scriptsctivate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.

Optional: copy `.env.example` to `.env` and set `HF_TOKEN` (from
https://huggingface.co/settings/tokens) to authenticate Hugging Face Hub requests --
avoids the "unauthenticated requests" rate-limit warning below and speeds up downloads.
Without it, the Aya/Dolly/PolyWrite loads below still work anonymously.

In [1]:
import prompts
import languages

print(f"{len(languages.LANGUAGES)} languages in the registry:")
for key, info in languages.LANGUAGES.items():
    sources = [s for s in ("aya", "dolly", "polywrite") if s in info]
    print(f"  {key:14s} ({info['name']:16s}) monolingual sources: {', '.join(sources) or '(none -- crosslingual target only)'}")


18 languages in the registry:
  acholi         (Acholi          ) monolingual sources: polywrite
  amharic        (Amharic         ) monolingual sources: aya, dolly, polywrite
  chichewa       (Chichewa        ) monolingual sources: aya, polywrite
  fongbe         (Fongbe          ) monolingual sources: polywrite
  ga             (Ga              ) monolingual sources: polywrite
  hausa          (Hausa           ) monolingual sources: aya, dolly, polywrite
  igbo           (Igbo            ) monolingual sources: aya, dolly, polywrite
  kinyarwanda    (Kinyarwanda     ) monolingual sources: polywrite
  lingala        (Lingala         ) monolingual sources: polywrite
  malagasy       (Malagasy        ) monolingual sources: aya, dolly, polywrite
  ndebele        (Ndebele         ) monolingual sources: polywrite
  sepedi         (Northern Sotho  ) monolingual sources: aya, dolly, polywrite
  shona          (Shona           ) monolingual sources: aya, dolly, polywrite
  swahili        (Swah

### Generate candidate pools for manual curation (Aya, Dolly, ShareGPT)

Run this once whenever you want a fresh candidate pool (or skip straight to **Build all
sources** below if you've already curated files from a previous run).

This downloads Aya, Dolly and the raw `RyokoAI/ShareGPT52K` scrape and filters each
(Aya and ShareGPT to 5-20 word prompts, ShareGPT additionally keeping English-only
turns since crosslingual prompts need an English base instruction, Dolly dropping any
prompt with a "Context:" block and keeping only >=5 word ones), then samples a
candidate pool per source:
- Aya: one CSV per language under `candidates/aya_candidates/{language}.csv` (200 candidates each)
- Dolly: one CSV per language under `candidates/dolly_candidates/{language}.csv` (250 candidates each)
- ShareGPT: a single pool at `candidates/sharegpt_candidates.csv` (400 candidates)

**Stop after running the cell below and manually curate before continuing:**
1. Open each candidate CSV, delete the rows you don't want.
2. Save Aya's pruned files to `candidates/aya_curated/{language}.csv` (100 prompts/language).
3. Save Dolly's pruned files to `candidates/dolly_curated/{language}.csv` (150 prompts/language).
4. Save ShareGPT's pruned file to `candidates/sharegpt_curated.csv` (200 prompts).

The **Build all sources** cell below reads only from the curated paths -- it'll raise a
clear `FileNotFoundError` telling you what's missing if you run it before curating.

In [2]:
prompts.generate_aya_candidates()
prompts.generate_dolly_candidates()
prompts.generate_sharegpt_candidates()
None

  Saved 200 candidates to candidates/aya_candidates\amharic.csv -- review and prune to 100, then save as candidates/aya_curated\amharic.csv
  Saved 200 candidates to candidates/aya_candidates\chichewa.csv -- review and prune to 100, then save as candidates/aya_curated\chichewa.csv
  Saved 200 candidates to candidates/aya_candidates\hausa.csv -- review and prune to 100, then save as candidates/aya_curated\hausa.csv
  Saved 200 candidates to candidates/aya_candidates\igbo.csv -- review and prune to 100, then save as candidates/aya_curated\igbo.csv
  Saved 200 candidates to candidates/aya_candidates\malagasy.csv -- review and prune to 100, then save as candidates/aya_curated\malagasy.csv
  [WARN] aya_candidates/sepedi: requested 200, only 69 available -- using all of them
  Saved 69 candidates to candidates/aya_candidates\sepedi.csv -- review and prune to 100, then save as candidates/aya_curated\sepedi.csv
  Saved 200 candidates to candidates/aya_candidates\shona.csv -- review and prune t

old/sg_52k.json: reconstructing file:   0%|          |  0.00B / 1.01GB            

old/sg_52k.json: downloading bytes:           |  0.00B            

c:\Users\johnu\Projects\Language Confusion\african-language-confusion\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\johnu\.cache\huggingface\hub\datasets--RyokoAI--ShareGPT52K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Generating train split: 0 examples [00:00, ? examples/s]

  crosslingual_sharegpt_candidates: 43145 kept, 84368 dropped for length, 11189 dropped as non-English
  Saved 400 candidates to candidates/sharegpt_candidates.csv -- review and prune to 200, then save as candidates/sharegpt_curated.csv


### Build all sources

This runs the active `LOADERS` (Aya, Dolly, PolyWrite, ShareGPT), filters to our
language registry, samples up to the target count per language (warning -- not silently
truncating -- when a language has fewer prompts available than requested), and writes
the CSVs. Aya, Dolly and ShareGPT read from the curated files produced in the previous
section instead of hitting Hugging Face directly, so this cell will fail with a
`FileNotFoundError` if you haven't generated and curated those yet. Takes a few minutes,
mostly spent downloading PolyWrite (~35k rows).


In [2]:
df = prompts.build_all_prompts()
prompts.save_test_sets(df, out_dir="prompts")
df.shape


Loading aya...
Loading dolly...
Loading polywrite...


Resolving data files:   0%|          | 0/240 [00:00<?, ?it/s]

  [WARN] polywrite/ga: requested 100, only 74 available -- using all of them
Loading afriqa...
Loading crosslingual_okapi...
Loading crosslingual_sharegpt...
  crosslingual_sharegpt_base: 55365 English turns kept, 3376 non-English turns dropped
Saved 11774 prompts to prompts/ (2 tasks, 6 sources, 18 languages)


(11774, 5)

### Prompt Statistics

In [3]:
summary = (
    df.groupby(["task", "source", "language"])
    .size()
    .rename("n_prompts")
    .reset_index()
    .pivot_table(index=["task", "source"], values="n_prompts", aggfunc=["sum", "mean", "min", "max"])
)
summary


sum         mean       min       max
                       n_prompts    n_prompts n_prompts n_prompts
task         source                                              
crosslingual okapi          1800   100.000000       100       100
             sharegpt       3600   200.000000       200       200
monolingual  afriqa         8000  1000.000000      1000      1000
             aya            1000   100.000000       100       100
             dolly          1600   200.000000       200       200
             polywrite      1774    98.555556        74       100

### Spot-check a few rendered prompts per source

In [3]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

for source in df["source"].unique():
    sample = df[df["source"] == source].sample(2, random_state=0)
    print(f"=== {source} ===")
    for _, row in sample.iterrows():
        print(f"[{row['language']}] {row['prompt']}")
    print()


=== aya ===
[yoruba] Àṣà ilẹ̀ Áfríkà
[wolof] Fann mooy gëblagréewum Inde?

=== dolly ===
[shona] Sarudza kana izvi zvingava zvinobetsera kana kuti zvisingabetseri kuti mudzidzi wechikoro chapamusoro azviise muhomwe yake. Mabhuku, bhuku rokudzidza, rambi repatafura, homwe yemapenzura, bhora rokumahombekombe, mutsago, laptop.
[hausa] Menene laser kuma wanene ya ƙirƙira shi?
Context:Laser na'ura ce da ke fitar da haske ta hanyar aiwatar da fadada gani wanda ya danganci fitowar fitowar hasken lantarki. Kalmar laser wani abu ne wanda ya samo asali ne a matsayin acronym don fadada haske ta hanyar motsawar radiation. An gina laser na farko a 1960 da Theodore Maiman a Hughes Research Laboratories, bisa ga aikin da Charles H. Townes da Arthur Leonard Schawlow suka yi.  Laser ya bambanta da sauran tushen haske a yadda yake fitar da haske da ke da daidaito. Haɗin sararin samaniya yana ba da damar mayar da hankali ga laser zuwa wani wuri mai mahimmanci, yana ba da damar aikace-aikace kamar yankan 